# Step 5: evaluating the step-4 fixes

Branches chosen in `support_diagnostics.ipynb`:
- **CIFAR-100-C → row 2 (contamination).** α-weighted support label purity was 48.8% (< 50%). Fix: `RamenClean`, $\alpha_{ij} \leftarrow \alpha_{ij}\,a_j^{\gamma}$, where $a_j$ is the share of $j$'s own α-weighted neighbours (excluding $j$) that lie in $j$'s queue. Offline, $\gamma$ = 1 / 2 raised that purity to 65.9% / 76.6%.
- **CIFAR-10-C → row 5 (nothing fired).** Normalized-SGD ablation `RamenNSGD`. Its L2 step length of 0.3 was the best of {0.01, …, 1} on a 200-per-corruption subset. Matching SignSGD's L2 step length ($\mathrm{lr}\sqrt D$) diverges.
- **`fp32prio`:** Ramen with fp32 cache priorities and no other change. It isolates the fp16 overflow that freezes full queues after 65,520 samples. Every other variant keeps Ramen's fp16 priorities, so they compare like for like with Ramen.

All runs use Ramen's hyperparameters (`cfg/<dataset>/Ramen.yaml`) and the same seeded stream. Code: `src/methods/RamenFix.py`; runs: `run_fixes.py`.

**Datasets:** only CIFAR-10-C and CIFAR-100-C are on this machine. ImageNet-C (5K) and DomainNet are not downloaded, and ViT-L/14 needs about 30 GB, more than the 24 GB GPUs.

In [1]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

import pandas as pd
from IPython.display import display
from ramen_analysis import make_experiment
import support_analysis as S
from run_fixes import VARIANTS

pd.set_option('display.width', 220)

/home/priyank1/miniconda3/envs/RAMEN/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASETS = ['CIFAR10C', 'CIFAR100C']
FIXES = list(VARIANTS)
exps, sls = {}, {}
for d in DATASETS:
    exps[d] = make_experiment(d).load()
    exps[d].evaluate()
    exps[d].run('RamenLog')
    for name, (algo, cfg) in VARIANTS.items():
        exps[d].run(algo, name=name, cfg=cfg)   # cached by run_fixes.py
    sls[d] = S.SupportLog(exps[d])

CIFAR10C: 15 corruptions x 10000 images, 10 classes, model=clip_vitbase32, mode=mixed
NoAdapt: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR10C_clip_vitbase32_mixed_nall_bs100_s0_NoAdapt.pkl
Ramen: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR10C_clip_vitbase32_mixed_nall_bs100_s0_Ramen.pkl
RamenLog: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR10C_clip_vitbase32_mixed_nall_bs100_s0_RamenLog.pkl
clean_g1: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR10C_clip_vitbase32_mixed_nall_bs100_s0_clean_g1.pkl
clean_g2: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR10C_clip_vitbase32_mixed_nall_bs100_s0_clean_g2.pkl
fp32prio: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR10C_clip_vitbase32_mixed_nall_bs100_s0_fp32prio.pkl
nsgd: loaded cached results from /

CIFAR100C: 15 corruptions x 10000 images, 100 classes, model=clip_vitbase16, mode=mixed
NoAdapt: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR100C_clip_vitbase16_mixed_nall_bs100_s0_NoAdapt.pkl


Ramen: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR100C_clip_vitbase16_mixed_nall_bs100_s0_Ramen.pkl
RamenLog: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR100C_clip_vitbase16_mixed_nall_bs100_s0_RamenLog.pkl
clean_g1: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR100C_clip_vitbase16_mixed_nall_bs100_s0_clean_g1.pkl
clean_g2: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR100C_clip_vitbase16_mixed_nall_bs100_s0_clean_g2.pkl
fp32prio: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR100C_clip_vitbase16_mixed_nall_bs100_s0_fp32prio.pkl
nsgd: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR100C_clip_vitbase16_mixed_nall_bs100_s0_nsgd.pkl


## Accuracy per corruption
The `… − Ramen` columns are the change from Ramen, in points.

In [3]:
for d in DATASETS:
    print(d)
    display(S.accuracy_by_domain(exps[d], ['NoAdapt', 'Ramen'] + FIXES))

CIFAR10C


,NoAdapt,Ramen,clean_g1,clean_g2,fp32prio,nsgd,clean_g1 - Ramen,clean_g2 - Ramen,fp32prio - Ramen,nsgd - Ramen
gaussian_noise,38.66,60.84,59.54,57.69,60.84,58.05,-1.30,-3.15,0.00,-2.79
shot_noise,42.59,63.14,62.38,60.60,63.08,59.77,-0.76,-2.54,-0.06,-3.37
impulse_noise,42.75,56.27,54.65,52.82,56.27,53.12,-1.62,-3.45,0.00,-3.15
defocus_blur,72.61,77.12,77.36,77.25,77.29,74.81,0.24,0.13,0.17,-2.31
glass_blur,43.91,64.01,64.21,64.35,63.81,57.79,0.20,0.34,-0.20,-6.22
motion_blur,66.68,77.55,77.82,77.51,77.33,71.85,0.27,-0.04,-0.22,-5.70
zoom_blur,71.69,79.27,79.48,79.35,79.33,75.85,0.21,0.08,0.06,-3.42
snow,73.78,80.65,80.49,80.52,80.69,77.80,-0.16,-0.13,0.04,-2.85
frost,75.69,80.16,80.44,80.36,80.26,77.22,0.28,0.20,0.10,-2.94
fog,68.85,77.93,77.98,77.69,78.03,74.63,0.05,-0.24,0.10,-3.30


CIFAR100C


,NoAdapt,Ramen,clean_g1,clean_g2,fp32prio,nsgd,clean_g1 - Ramen,clean_g2 - Ramen,fp32prio - Ramen,nsgd - Ramen
gaussian_noise,22.83,30.37,29.26,26.76,30.32,25.26,-1.11,-3.61,-0.05,-5.11
shot_noise,24.42,31.98,31.60,29.28,31.93,27.27,-0.38,-2.70,-0.05,-4.71
impulse_noise,29.58,42.94,41.64,39.71,43.01,34.10,-1.30,-3.23,0.07,-8.84
defocus_blur,43.56,52.60,51.06,48.57,52.47,45.25,-1.54,-4.03,-0.13,-7.35
glass_blur,20.16,28.95,28.59,26.32,29.00,23.76,-0.36,-2.63,0.05,-5.19
motion_blur,43.60,52.38,51.59,49.09,52.35,45.70,-0.79,-3.29,-0.03,-6.68
zoom_blur,48.64,55.74,54.11,51.78,55.65,50.34,-1.63,-3.96,-0.09,-5.40
snow,48.91,55.30,54.97,53.78,55.44,51.43,-0.33,-1.52,0.14,-3.87
frost,50.48,54.40,54.54,52.60,54.56,51.96,0.14,-1.80,0.16,-2.44
fog,41.77,52.84,52.73,51.15,52.67,48.59,-0.11,-1.69,-0.17,-4.25


## Transition table, before (Ramen) and after
As % of samples. *right → wrong* = harmful, *wrong → right* = helpful.

In [4]:
for d in DATASETS:
    print(d)
    display(S.transition_compare(sls[d], ['Ramen'] + FIXES))

CIFAR10C


,Ramen,clean_g1,clean_g2,fp32prio,nsgd
cell,,,,,
right -> right,55.70,55.47,55.05,55.85,55.38
right -> wrong,5.67,5.89,6.31,5.52,5.99
wrong -> right,16.91,16.85,16.75,16.81,13.39
wrong -> same wrong,12.00,11.59,11.14,12.15,15.97
wrong -> other wrong,9.72,10.19,10.74,9.67,9.27


CIFAR100C


,Ramen,clean_g1,clean_g2,fp32prio,nsgd
cell,,,,,
right -> right,32.56,32.18,31.43,32.71,31.62
right -> wrong,4.53,4.91,5.66,4.38,5.47
wrong -> right,13.53,13.19,11.75,13.38,8.73
wrong -> same wrong,20.20,19.88,20.35,20.65,26.41
wrong -> other wrong,29.18,29.84,30.81,28.88,27.77


Per corruption: change in each cell for the contamination fix, in points relative to Ramen.

In [5]:
for d in DATASETS:
    print(d, ': clean_g2 - Ramen')
    display((S.transition_table(sls[d], 'clean_g2', by_domain=True) - S.transition_table(sls[d], 'Ramen', by_domain=True)).round(2))

CIFAR10C : clean_g2 - Ramen


cell,right -> right,right -> wrong,wrong -> right,wrong -> same wrong,wrong -> other wrong
gaussian_noise,-1.38,1.38,-1.77,-1.40,3.17
shot_noise,-1.21,1.21,-1.33,-1.19,2.52
impulse_noise,-1.85,1.85,-1.60,-1.72,3.32
defocus_blur,-0.05,0.05,0.18,-0.49,0.31
glass_blur,-0.81,0.81,1.15,-1.76,0.61
motion_blur,-0.31,0.31,0.27,-0.70,0.43
zoom_blur,-0.16,0.16,0.24,-0.81,0.57
snow,-0.20,0.20,0.07,-0.32,0.25
frost,-0.10,0.10,0.30,-0.49,0.19
fog,-0.29,0.29,0.05,-0.53,0.48


CIFAR100C : clean_g2 - Ramen


cell,right -> right,right -> wrong,wrong -> right,wrong -> same wrong,wrong -> other wrong
gaussian_noise,-1.97,1.97,-1.64,-3.48,5.12
shot_noise,-1.83,1.83,-0.87,-3.78,4.65
impulse_noise,-0.79,0.79,-2.44,1.48,0.96
defocus_blur,-1.42,1.42,-2.61,0.88,1.73
glass_blur,-0.68,0.68,-1.95,2.20,-0.25
motion_blur,-1.43,1.43,-1.86,0.70,1.16
zoom_blur,-2.01,2.01,-1.95,-0.03,1.98
snow,-0.88,0.88,-0.64,0.20,0.44
frost,-0.97,0.97,-0.83,0.32,0.51
fog,-0.34,0.34,-1.35,0.11,1.24


## What the fixes did inside the method
`own-queue alpha share` is the share of a query's α mass that comes from its own pseudo-label queue. If reweighting increases it a lot, the fix is partly acting as a tilt toward the query's own pseudo-label, the option-2 effect, rather than only removing contaminated entries. `inserted %` for `fp32prio` shows the cache no longer freezes.

In [6]:
for d in DATASETS:
    print(d)
    display(S.fix_stats(exps[d], ['clean_g1', 'clean_g2', 'fp32prio']))

CIFAR10C


,inserted %,mean agreement a,own-queue alpha share before %,own-queue alpha share after %
clean_g1,90.953,0.202,22.021,26.465
clean_g2,90.953,0.202,22.021,30.236
fp32prio,100.000,0.202,22.133,22.133


CIFAR100C


,inserted %,mean agreement a,own-queue alpha share before %,own-queue alpha share after %
clean_g1,82.267,0.038,7.153,9.731
clean_g2,82.267,0.038,7.153,10.108
fp32prio,100.000,0.037,7.579,7.579
